# 01 — Dataset Validation and Manifest Generation

This notebook uses your exact local paths.

**Project root**  
`D:\PAPERS\SPEECH\low_snr_speech_enhancement`

**VoiceBank+DEMAND root**  
`D:\PAPERS\SPEECH\low_snr_speech_enhancement\DS_10283_2791`

**LibriSpeech test-clean**  
`D:\PAPERS\SPEECH\low_snr_speech_enhancement\test-clean\LibriSpeech\test-clean`

The DNS noise path is deliberately left unset because you have not supplied the real extracted location yet.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")
VOICEBANK_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement\DS_10283_2791")
OOD_CLEAN_DIR = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement\test-clean\LibriSpeech\test-clean")
OOD_NOISE_DIR = None

MANIFEST_DIR = PROJECT_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_OOD_DIR = PROJECT_ROOT / "generated_low_snr_test"

SR = 16000
SEED = 2026
AUDIO_EXTS = {".wav", ".flac"}

print("PROJECT_ROOT exists :", PROJECT_ROOT.exists(), PROJECT_ROOT)
print("VOICEBANK_ROOT exists:", VOICEBANK_ROOT.exists(), VOICEBANK_ROOT)
print("OOD_CLEAN_DIR exists :", OOD_CLEAN_DIR.exists(), OOD_CLEAN_DIR)

## Auto-detect the VoiceBank+DEMAND folders

In [ ]:
def find_dir(root, exact_name):
    matches = [p for p in Path(root).rglob(exact_name) if p.is_dir()]
    if not matches:
        return None
    if len(matches) > 1:
        print(f"WARNING: multiple matches for {exact_name}:")
        for m in matches:
            print("  ", m)
    return matches[0]

expected = {
    "train_clean": "clean_trainset_28spk_wav",
    "train_noisy": "noisy_trainset_28spk_wav",
    "test_clean": "clean_testset_wav",
    "test_noisy": "noisy_testset_wav",
}

VOICEBANK_TRAIN_CLEAN = find_dir(VOICEBANK_ROOT, expected["train_clean"])
VOICEBANK_TRAIN_NOISY = find_dir(VOICEBANK_ROOT, expected["train_noisy"])
VOICEBANK_TEST_CLEAN  = find_dir(VOICEBANK_ROOT, expected["test_clean"])
VOICEBANK_TEST_NOISY  = find_dir(VOICEBANK_ROOT, expected["test_noisy"])

paths = {
    "VOICEBANK_TRAIN_CLEAN": VOICEBANK_TRAIN_CLEAN,
    "VOICEBANK_TRAIN_NOISY": VOICEBANK_TRAIN_NOISY,
    "VOICEBANK_TEST_CLEAN": VOICEBANK_TEST_CLEAN,
    "VOICEBANK_TEST_NOISY": VOICEBANK_TEST_NOISY,
}

for name, path in paths.items():
    print(f"{name:24s} -> {path}")

missing = [name for name, path in paths.items() if path is None]
if missing:
    print("\\nTop-level content:")
    for p in sorted(VOICEBANK_ROOT.iterdir()):
        print("  ", p.name)
    raise FileNotFoundError(f"Missing required VoiceBank folders: {missing}")

print("\\nVoiceBank+DEMAND folders detected successfully.")

## Verify paired clean/noisy files

In [ ]:
def recursive_audio_map(root):
    items = {}
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            if p.stem in items:
                raise RuntimeError(f"Duplicate audio stem detected: {p.stem}")
            items[p.stem] = p.resolve()
    return items

train_clean_map = recursive_audio_map(VOICEBANK_TRAIN_CLEAN)
train_noisy_map = recursive_audio_map(VOICEBANK_TRAIN_NOISY)
test_clean_map  = recursive_audio_map(VOICEBANK_TEST_CLEAN)
test_noisy_map  = recursive_audio_map(VOICEBANK_TEST_NOISY)

print("Train clean:", len(train_clean_map))
print("Train noisy:", len(train_noisy_map))
print("Test clean :", len(test_clean_map))
print("Test noisy :", len(test_noisy_map))

assert set(train_clean_map) == set(train_noisy_map), "Training clean/noisy filenames do not match."
assert set(test_clean_map) == set(test_noisy_map), "Test clean/noisy filenames do not match."

print("Pairing check: PASS")

## Build deterministic train/validation/test manifests

In [ ]:
def speaker_from_stem(stem):
    return stem.split("_")[0] if "_" in stem else stem

train_all = pd.DataFrame([
    {
        "utt_id": stem,
        "speaker_id": speaker_from_stem(stem),
        "clean_path": str(train_clean_map[stem]),
        "noisy_path": str(train_noisy_map[stem]),
    }
    for stem in sorted(train_clean_map)
])

test_df = pd.DataFrame([
    {
        "utt_id": stem,
        "speaker_id": speaker_from_stem(stem),
        "clean_path": str(test_clean_map[stem]),
        "noisy_path": str(test_noisy_map[stem]),
        "split": "test",
    }
    for stem in sorted(test_clean_map)
])

speakers = sorted(train_all["speaker_id"].unique())
n_val_speakers = max(1, round(len(speakers) * 0.10))
ordered = sorted(speakers, key=lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())
val_speakers = set(ordered[:n_val_speakers])

val_df = train_all[train_all["speaker_id"].isin(val_speakers)].copy()
train_df = train_all[~train_all["speaker_id"].isin(val_speakers)].copy()
train_df["split"] = "train"
val_df["split"] = "val"

voicebank_manifest_dir = MANIFEST_DIR / "voicebank"
voicebank_manifest_dir.mkdir(parents=True, exist_ok=True)

train_csv = voicebank_manifest_dir / "train.csv"
val_csv = voicebank_manifest_dir / "val.csv"
test_csv = voicebank_manifest_dir / "test.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

assert set(train_df.utt_id).isdisjoint(set(val_df.utt_id))
assert set(train_df.utt_id).isdisjoint(set(test_df.utt_id))
assert set(val_df.utt_id).isdisjoint(set(test_df.utt_id))
assert set(train_df.speaker_id).isdisjoint(set(val_df.speaker_id))
assert set(train_df.speaker_id).isdisjoint(set(test_df.speaker_id))
assert set(val_df.speaker_id).isdisjoint(set(test_df.speaker_id))

print("Training utterances  :", len(train_df))
print("Validation utterances:", len(val_df))
print("Test utterances      :", len(test_df))
print("Validation speakers  :", sorted(val_speakers))
print("\\nSaved manifests:")
print(train_csv)
print(val_csv)
print(test_csv)

display(train_df.head())

## Inspect real VoiceBank audio metadata

In [ ]:
def audio_info(path):
    info = sf.info(path)
    return {
        "path": str(path),
        "samplerate": info.samplerate,
        "frames": info.frames,
        "channels": info.channels,
        "duration_sec": info.frames / info.samplerate
    }

audio_metadata = pd.DataFrame([
    audio_info(Path(train_df.iloc[0].clean_path)),
    audio_info(Path(train_df.iloc[0].noisy_path)),
    audio_info(Path(test_df.iloc[0].clean_path)),
    audio_info(Path(test_df.iloc[0].noisy_path)),
])

display(audio_metadata)

## Validate LibriSpeech test-clean

In [ ]:
libri_files = sorted(OOD_CLEAN_DIR.rglob("*.flac"))

print("LibriSpeech directory:", OOD_CLEAN_DIR)
print("Folder exists:", OOD_CLEAN_DIR.exists())
print("FLAC files found:", len(libri_files))

if not libri_files:
    raise FileNotFoundError("No .flac files found in the configured LibriSpeech test-clean path.")

print("\\nFirst five files:")
for f in libri_files[:5]:
    print(" ", f)

print("\\nFirst-file metadata:")
print(sf.info(libri_files[0]))

## DNS noise path

The OOD mixture generator remains disabled until you provide the real DNS noise folder.

When the dataset is available, replace `None` with the exact extracted path and rerun this cell.

In [ ]:
OOD_NOISE_DIR = None

if OOD_NOISE_DIR is None:
    DNS_READY = False
    dns_noise_files = []
    print("DNS noise path not configured. OOD mixture generation will be skipped.")
else:
    OOD_NOISE_DIR = Path(OOD_NOISE_DIR)
    dns_noise_files = sorted(
        p for p in OOD_NOISE_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS
    )
    DNS_READY = OOD_NOISE_DIR.exists() and len(dns_noise_files) > 0
    print("DNS folder exists:", OOD_NOISE_DIR.exists())
    print("DNS audio files:", len(dns_noise_files))

## Define fixed severe-SNR mixture functions

In [ ]:
def read_mono(path, target_sr=16000):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr).astype(np.float32)
    return wav

def repeat_or_crop(noise, length, rng):
    if len(noise) >= length:
        start = int(rng.integers(0, len(noise) - length + 1))
        return noise[start:start+length]
    if len(noise) == 0:
        raise ValueError("Empty noise file")
    reps = int(np.ceil(length / len(noise)))
    return np.tile(noise, reps)[:length]

def active_rms(x, eps=1e-10):
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0:
        return 0.0
    frame = int(0.020 * SR)
    hop = int(0.010 * SR)
    if len(x) < frame:
        return float(np.sqrt(np.mean(x*x) + eps))
    vals = []
    for start in range(0, len(x)-frame+1, hop):
        seg = x[start:start+frame]
        vals.append(np.sqrt(np.mean(seg*seg) + eps))
    vals = np.asarray(vals)
    threshold = max(vals.max()*0.01, eps)
    active = vals[vals >= threshold]
    return float(active.mean() if len(active) else vals.mean())

def mix_at_snr(clean, noise, snr_db, eps=1e-10):
    clean = np.asarray(clean, dtype=np.float32)
    noise = np.asarray(noise, dtype=np.float32)

    c_rms = active_rms(clean, eps)
    n_rms = float(np.sqrt(np.mean(noise.astype(np.float64)**2) + eps))

    if c_rms <= eps:
        raise ValueError("Near-silent clean speech")
    if n_rms <= eps:
        raise ValueError("Near-silent noise")

    target_noise_rms = c_rms / (10.0 ** (snr_db / 20.0))
    scaled_noise = noise * (target_noise_rms / (n_rms + eps))
    noisy = clean + scaled_noise

    peak = max(
        float(np.max(np.abs(clean))),
        float(np.max(np.abs(scaled_noise))),
        float(np.max(np.abs(noisy))),
        1e-12
    )
    if peak > 0.99:
        scale = 0.99 / peak
        clean *= scale
        scaled_noise *= scale
        noisy *= scale

    realized_snr = 10.0*np.log10(
        (active_rms(clean, eps)**2 + eps) /
        (np.mean(scaled_noise.astype(np.float64)**2) + eps)
    )
    return clean, scaled_noise, noisy, float(realized_snr)

print("Mixture functions defined.")

## Generate the frozen low-SNR OOD set only when DNS is ready

In [ ]:
if not DNS_READY:
    print("SKIPPED: DNS noise is not configured.")
else:
    SNRS = [-10, -5, 0, 5, 10]
    MAX_CLEAN = min(200, len(libri_files))

    clean_out = GENERATED_OOD_DIR / "clean"
    noise_out = GENERATED_OOD_DIR / "noise"
    noisy_out = GENERATED_OOD_DIR / "noisy"
    for d in [clean_out, noise_out, noisy_out]:
        d.mkdir(parents=True, exist_ok=True)

    master_rng = np.random.default_rng(SEED)
    rows = []

    for ci, clean_path in enumerate(tqdm(libri_files[:MAX_CLEAN], desc="Generating OOD mixtures")):
        clean0 = read_mono(clean_path, SR)
        if len(clean0) == 0:
            continue

        for target_snr in SNRS:
            noise_path = dns_noise_files[int(master_rng.integers(0, len(dns_noise_files)))]
            key = f"{SEED}|{clean_path}|{noise_path}|{target_snr}".encode("utf-8")
            local_seed = int(hashlib.sha256(key).hexdigest()[:8], 16)
            local_rng = np.random.default_rng(local_seed)

            noise0 = read_mono(noise_path, SR)
            noise0 = repeat_or_crop(noise0, len(clean0), local_rng)

            clean, noise, noisy, realized = mix_at_snr(clean0.copy(), noise0, target_snr)

            utt_id = f"{clean_path.stem}__snr{target_snr:+d}__{ci:05d}"
            clean_file = clean_out / f"{utt_id}.wav"
            noise_file = noise_out / f"{utt_id}.wav"
            noisy_file = noisy_out / f"{utt_id}.wav"

            sf.write(clean_file, clean, SR, subtype="FLOAT")
            sf.write(noise_file, noise, SR, subtype="FLOAT")
            sf.write(noisy_file, noisy, SR, subtype="FLOAT")

            rows.append({
                "utt_id": utt_id,
                "clean_source": str(clean_path.resolve()),
                "noise_source": str(noise_path.resolve()),
                "clean_path": str(clean_file.resolve()),
                "noise_path": str(noise_file.resolve()),
                "noisy_path": str(noisy_file.resolve()),
                "target_snr_db": target_snr,
                "realized_snr_db": realized,
                "seed": local_seed,
                "sample_rate": SR,
            })

    ood_df = pd.DataFrame(rows)
    ood_dir = MANIFEST_DIR / "low_snr_ood"
    ood_dir.mkdir(parents=True, exist_ok=True)
    ood_csv = ood_dir / "manifest.csv"
    ood_df.to_csv(ood_csv, index=False)

    display(ood_df.head())
    display(ood_df.groupby("target_snr_db").size().rename("count"))
    print("Mean absolute SNR error:",
          float(np.mean(np.abs(ood_df.realized_snr_db - ood_df.target_snr_db))))
    print("Saved:", ood_csv)

## Final readiness summary

In [ ]:
summary = {
    "project_root": str(PROJECT_ROOT),
    "voicebank_root": str(VOICEBANK_ROOT),
    "voicebank_train_manifest": str(MANIFEST_DIR / "voicebank" / "train.csv"),
    "voicebank_val_manifest": str(MANIFEST_DIR / "voicebank" / "val.csv"),
    "voicebank_test_manifest": str(MANIFEST_DIR / "voicebank" / "test.csv"),
    "librispeech_test_clean": str(OOD_CLEAN_DIR),
    "dns_ready": bool(DNS_READY),
    "ood_manifest": str(MANIFEST_DIR / "low_snr_ood" / "manifest.csv") if DNS_READY else None,
}

path_record = MANIFEST_DIR / "dataset_paths.json"
path_record.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("\\nVoiceBank manifests: READY")
print("LibriSpeech: READY")
print("DNS OOD:", "READY" if DNS_READY else "WAITING FOR REAL PATH")
print("Saved path record:", path_record)